In [1]:
from datetime import date

import pandas as pd
from sqlalchemy import func, or_

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN,               # applications
    TLS202_APPLN_TITLE,         # titles
    TLS203_APPLN_ABSTR,         # abstracts
    TLS206_PERSON,              # persons (applicants and inventors)
    TLS207_PERS_APPLN,          # person <-> application links
    TLS209_APPLN_IPC,           # IPC classification
    TLS224_APPLN_CPC,           # CPC classification
    TLS231_INPADOC_LEGAL_EVENT, # legal events
)

In [2]:
patstat = PatstatClient()          # default: env="TEST"; use PatstatClient(env="PROD") for the full database
db = patstat.orm()                 # SQLAlchemy session used by all ORM queries below

This client instance is currently configured to use a test dataset with reduced number of publications (~10K).
Use PatstatClient(env='PROD') to use the complete PATSTAT dataset (>140M publications).
Use PatstatClient(env='TEST') to use the test dataset and avoid displaying this warning



In [3]:
CANDIDATE_DRONE_KEYWORDS = (
    # General UAV terminology
    "unmanned aerial vehicle",
    "unmanned aerial system",
    "unmanned aircraft",
    "uncrewed aerial vehicle",
    "uncrewed aircraft",
    "unmanned air vehicle",
    "remotely piloted aircraft",
    "remotely piloted aerial vehicle",
    "remotely controlled aircraft",
    "unmanned flying vehicle",

    # Common acronyms
    "UAV",
    "RPAS",

    # Drone types
    "quadcopter",
    "quadrotor",
    "quad-rotor",
    "hexacopter",
    "hexarotor",
    "octocopter",

    # Additional UAV terminology
    "aerial drone",
    "flying drone",
    "drone aircraft",
    "fixed-wing drone",
    "flying robot",
    "micro air vehicle",
)


In [4]:
#"remote-controlled aircraft","remote controlled aircraft",

DRONE_KEYWORDS = (
    # "unmanned/unpiloted" + aircraft/aerial (both papers)
    "unmanned aerial", "unmanned aircraft", "unmanned air vehicle",
    "unpiloted aerial", "unpiloted aircraft", "unpiloted air vehicle",
    # remote* within 1 word of aircraft (DTI)
    "remotely piloted aircraft",
    "micro air vehicle",                                    # DTI
    # rotor types (Gorry)
    "quadrotor", "quadrocopter", "quadcopter", "multicopter", "multirotor", "multi-rotor",
    "hexacopter", "octocopter",
    " uav", "uav ",
    "drone",                                                # contested: keep only if hand-labelling supports it
)
#these are  possible keywords found in scientific paper

In [5]:
DRONE_IPC_CODES = (
    "B64U",
)

In [6]:
DRONE_CPC_CODES = (
    "B64U",
    "B64C39/024",
)

In [7]:
def keyword_condition(terms):
    """OR-condition: any term appears in the title or in the abstract (case-insensitive)."""
    title = func.lower(TLS202_APPLN_TITLE.appln_title)
    abstract = func.lower(TLS203_APPLN_ABSTR.appln_abstract)
    return or_(
        *(title.like(f"%{t.lower()}%") for t in terms),
        *(abstract.like(f"%{t.lower()}%") for t in terms),
    )


def symbol_prefix_condition(symbol_column, codes):
    """OR-condition: the blank-stripped classification symbol starts with one of the codes."""
    symbol = func.replace(symbol_column, " ", "")
    return or_(*(symbol.like(code.replace(" ", "") + "%") for code in codes))


# Application-level fields retrieved by the search queries.
# These fields describe each application; they do not change the search criteria.
APPLN_COLUMNS = (
    TLS201_APPLN.appln_id,              # Unique application identifier
    TLS201_APPLN.docdb_family_id,       # Links applications belonging to the same DOCDB family
    TLS201_APPLN.appln_auth,            # Filing authority or patent office
    TLS201_APPLN.appln_nr,              # Application number
    TLS201_APPLN.appln_kind,            # Application kind code
    TLS201_APPLN.appln_filing_date,     # Filing date of this application
    TLS201_APPLN.earliest_filing_year,  # Earliest filing year associated with this application
    TLS201_APPLN.granted,               # Grant indicator: "Y" means granted
)


def query_by_keywords(*columns):
    """Applications whose title or abstract contains an Drone keyword."""
    return (
        db.query(*columns)
        .select_from(TLS201_APPLN)
        .outerjoin(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
        .outerjoin(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id)
        .filter(keyword_condition(DRONE_KEYWORDS))
    )


def query_by_ipc(*columns):
    """Applications with at least one IPC symbol in the Drone list."""
    return (
        db.query(*columns)
        .select_from(TLS201_APPLN)
        .join(TLS209_APPLN_IPC, TLS201_APPLN.appln_id == TLS209_APPLN_IPC.appln_id)
        .filter(symbol_prefix_condition(TLS209_APPLN_IPC.ipc_class_symbol, DRONE_IPC_CODES))
    )


def query_by_cpc(*columns):
    """Applications with at least one CPC symbol in the Drone list."""
    return (
        db.query(*columns)
        .select_from(TLS201_APPLN)
        .join(TLS224_APPLN_CPC, TLS201_APPLN.appln_id == TLS224_APPLN_CPC.appln_id)
        .filter(symbol_prefix_condition(TLS224_APPLN_CPC.cpc_class_symbol, DRONE_CPC_CODES))
    )

In [8]:
#KEYWORD SEARCH

q_keywords = query_by_keywords(
    *APPLN_COLUMNS,
    TLS202_APPLN_TITLE.appln_title,
    TLS203_APPLN_ABSTR.appln_abstract,
).distinct()

df_keywords = patstat.df(q_keywords)
df_keywords.head(10)

,appln_id,docdb_family_id,appln_auth,appln_nr,appln_kind,appln_filing_date,earliest_filing_year,granted,appln_title,appln_abstract
0,486086024,60263566,RU,2016122581,A,2016-06-07,2016,Y,METHOD OF DETERMINING STATE OF ICE COVER,FIELD: measuring equipment.SUBSTANCE: using a ...
1,519747364,66175445,GR,20180100110,A,2018-03-08,2018,Y,FLOATING PLATFORM FOR MARITIME SURVEILLANCE AN...,The invention relates to an arrangement of one...
2,443291632,53758549,CA,2840823,A,2014-01-28,2014,N,VERTICAL TAKE-OFF AND LANDING ROADABLE AIRCRAFT,A vertical take-off and landing (VTOL) roadabl...
3,539038255,72709551,FR,2006719,A,2020-06-26,2020,N,Aérogyre télécommandé de mesure par établissem...,Aérogyre (B) sans pilote télécommandé de mesur...
4,566134233,80039055,RU,2021105888,A,2021-03-05,2021,Y,MOBILE AEROENERGOSTAT MODULE,FIELD: power engineering.SUBSTANCE: invention ...
5,534522371,60412625,RU,2018145854,A,2017-05-25,2016,Y,BLADE SCREW,FIELD: machine building.SUBSTANCE: blade screw...
6,570351315,80999538,FR,2200816,A,2022-01-31,2022,N,Procédé de navigation autonome d’un drone vola...,Procédé de navigation autonome d’un drone vola...
7,530889330,56108250,AU,2020202629,A,2020-04-17,2014,Y,Systems and methods for automatic vehicle imaging,Systems and methods are provided for automatic...
8,546391843,74666363,RU,2020125245,A,2019-12-04,2019,Y,LOW-TEMPERATURE SCANNING INFRARED ANALYZER OF ...,FIELD: measuring equipment.SUBSTANCE: inventio...
9,590349184,83459228,TW,111111954,A,2022-03-29,2021,N,Maintenance method for structure,The present invention provides a maintenance m...


In [9]:
#IPC SEARCH
q_ipc = query_by_ipc(*APPLN_COLUMNS, TLS209_APPLN_IPC.ipc_class_symbol).distinct()

df_ipc = patstat.df(q_ipc)
df_ipc.head()

,appln_id,docdb_family_id,appln_auth,appln_nr,appln_kind,appln_filing_date,earliest_filing_year,granted,ipc_class_symbol
0,513740868,48290734,US,201916265314,A,2019-02-01,2011,Y,B64U 10/25
1,513740868,48290734,US,201916265314,A,2019-02-01,2011,Y,B64U 20/10
2,556168280,49580728,US,202117233087,A,2021-04-16,2012,Y,B64U 101/00
3,567233887,50733318,US,202117520459,A,2021-11-05,2013,Y,B64U 10/13
4,567233887,50733318,US,202117520459,A,2021-11-05,2013,Y,B64U 80/60


In [10]:
#CPC SEARCH
q_cpc = query_by_cpc(*APPLN_COLUMNS, TLS224_APPLN_CPC.cpc_class_symbol).distinct()

df_cpc = patstat.df(q_cpc)
df_cpc.head(10)

,appln_id,docdb_family_id,appln_auth,appln_nr,appln_kind,appln_filing_date,earliest_filing_year,granted,cpc_class_symbol
0,52104346,24390749,US,59725632,A,1932-03-07,1932,Y,B64U 10/60
1,54372193,25514354,US,96850378,A,1978-12-11,1978,Y,B64U 10/60
2,46566715,22019089,US,5881587,A,1987-06-05,1987,Y,B64U 70/30
3,50237261,27320303,US,36900589,A,1989-06-20,1988,Y,B64U 10/60
4,52709811,24729242,US,68000096,A,1996-07-15,1996,Y,B64C 39/024
5,52709811,24729242,US,68000096,A,1996-07-15,1996,Y,B64U 10/10
6,52709811,24729242,US,68000096,A,1996-07-15,1996,Y,B64U 10/25
7,52709811,24729242,US,68000096,A,1996-07-15,1996,Y,B64U 30/20
8,52709811,24729242,US,68000096,A,1996-07-15,1996,Y,B64U 30/21
9,52709811,24729242,US,68000096,A,1996-07-15,1996,Y,B64U 30/294


In [11]:
# COMBINE THE SEARCH RESULTS

# Select application IDs found by at least one search strategy:
# keywords OR IPC codes OR CPC codes.
# UNION removes duplicate IDs across the three searches.
appln_id = TLS201_APPLN.appln_id.label("appln_id")

drone_ids = (
    query_by_keywords(appln_id)
    .union(
        query_by_ipc(appln_id),
        query_by_cpc(appln_id),
    )
    .subquery()
)

# Retrieve application details, titles and abstracts for the selected IDs.
# Outer joins preserve applications even when a title or abstract is missing.
q_drone = (
    db.query(
        *APPLN_COLUMNS,
        TLS202_APPLN_TITLE.appln_title,
        TLS203_APPLN_ABSTR.appln_abstract,
    )
    .select_from(TLS201_APPLN)
    .join(
        drone_ids,
        TLS201_APPLN.appln_id == drone_ids.c.appln_id,
    )
    .outerjoin(
        TLS202_APPLN_TITLE,
        TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id,
    )
    .outerjoin(
        TLS203_APPLN_ABSTR,
        TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id,
    )
    .distinct()  # Remove identical result rows
    .order_by(TLS201_APPLN.appln_id)
)

# Execute the query and store the results in a pandas DataFrame.
# This is still an application-level dataset, not a family-level dataset.
df_drone = patstat.df(q_drone)

df_drone.head(10)

,appln_id,docdb_family_id,appln_auth,appln_nr,appln_kind,appln_filing_date,earliest_filing_year,granted,appln_title,appln_abstract
0,24746,37809459,EP,06802616,A,2006-08-30,2005,N,MULTI-ROTOR WIND TURBINE SUPPORTED BY CONTINUO...,"Co-axial, multi-rotor wind turbines, producing..."
1,25546,39689224,EP,08157886,A,2008-06-09,2007,Y,Airborne manipulator unmanned aerial vehicle (...,A manipulator arm system (1) on a ducted air-f...
2,352060,33524069,EP,04017945,A,2004-07-29,2003,N,A flying machine,The flying machine (1) comprises a supporting ...
3,2755148,37809459,AU,2006284845,A,2006-08-30,2005,Y,Multi-rotor wind turbine supported by continuo...,"Co-axial, multi-rotor wind turbines, producing..."
4,4305581,25672538,CA,594383,A,1989-03-21,1987,Y,MONITORING SYSTEM FOR POWER LINES AND RIGHT-OF...,Application Of: Roosevelt A. Fernandes For: Mo...
5,5009380,37809459,CA,2620922,A,2006-08-30,2005,N,MULTI-ROTOR WIND TURBINE SUPPORTED BY CONTINUO...,"Co-axial, multi-rotor wind turbines, producing..."
6,6750614,26877844,CN,01807474,A,2001-02-12,2000,N,Remotely piloted aircraft,None
7,7611249,5221267,CN,98111274,A,1998-05-04,1998,N,Wind motor,"The present invention relates to a wind motor,..."
8,13609024,7942894,DE,20010668,U,2000-06-15,2000,Y,Windenergieanlage Multi-Rotor-Prinzip,None
9,14034049,11074674,DE,60119462,T,2001-09-10,2000,Y,Unbemannte mobile Vorrichtung,None


In [12]:
# How much does each strategy contribute? (counts of distinct applications)
pd.Series(
    {
        "Keywords": df_keywords["appln_id"].nunique(),
        "IPC": df_ipc["appln_id"].nunique(),
        "CPC": df_cpc["appln_id"].nunique(),
        "Total union": df_drone["appln_id"].nunique(),
    },
    name="Applications"
)

Keywords       2200
IPC             126
CPC            2190
Total union    3466
Name: Applications, dtype: int64

In [13]:
# INITIAL DATA QUALITY CHECKS
# These checks report potential issues without modifying or deleting any data.

# Compare the number of rows with the number of distinct applications.
# A difference indicates that some application IDs appear on multiple rows.
print("Total rows:", len(df_drone))
print("Distinct applications:", df_drone["appln_id"].nunique())

# A usable DOCDB family ID must be present and greater than zero.
valid_family = (
    df_drone["docdb_family_id"].notna()
    & (df_drone["docdb_family_id"] > 0)
)

print(
    "Distinct families with a valid ID:",
    df_drone.loc[valid_family, "docdb_family_id"].nunique(),
)

print(
    "Applications without a valid family ID:",
    df_drone.loc[~valid_family, "appln_id"].nunique(),
)

# Convert years to numeric values for consistent missing-value checks.
# PATSTAT uses 9999 for an unavailable year.
priority_year = pd.to_numeric(
    df_drone["earliest_filing_year"],
    errors="coerce",
)

missing_priority_year = (
    priority_year.isna()
    | (priority_year == 9999)
)

print(
    "Applications with an unavailable earliest filing year:",
    df_drone.loc[missing_priority_year, "appln_id"].nunique(),
)

Total rows: 3466
Distinct applications: 3466
Distinct families with a valid ID: 2469
Applications without a valid family ID: 0
Applications with an unavailable earliest filing year: 0


In [14]:
# IDENTIFY THE SEARCH STRATEGIES THAT RETRIEVED EACH APPLICATION
# Create a diagnostic copy without changing the original dataset.
from IPython.display import display

df_diagnostics = df_drone.copy()

# A value of True means that the application was found by that strategy.
df_diagnostics["found_by_keywords"] = (
    df_diagnostics["appln_id"].isin(df_keywords["appln_id"])
)

df_diagnostics["found_by_ipc"] = (
    df_diagnostics["appln_id"].isin(df_ipc["appln_id"])
)

df_diagnostics["found_by_cpc"] = (
    df_diagnostics["appln_id"].isin(df_cpc["appln_id"])
)


# Combine the flags into a readable label.
def get_retrieval_strategy(row):
    strategies = []

    if row["found_by_keywords"]:
        strategies.append("Keywords")
    if row["found_by_ipc"]:
        strategies.append("IPC")
    if row["found_by_cpc"]:
        strategies.append("CPC")

    return " + ".join(strategies)


df_diagnostics["retrieval_strategy"] = df_diagnostics.apply(
    get_retrieval_strategy,
    axis=1,
)

# Count distinct applications for each combination.
# Each application is counted in exactly one combination.
strategy_summary = (
    df_diagnostics
    .groupby("retrieval_strategy")["appln_id"]
    .nunique()
    .rename("n_applications")
    .sort_values(ascending=False)
    .reset_index()
)

display(strategy_summary)

# Show applications retrieved only by keywords.
# These are candidates for inspection, not automatically irrelevant results.
keyword_only = df_diagnostics.loc[
    df_diagnostics["found_by_keywords"]
    & ~df_diagnostics["found_by_ipc"]
    & ~df_diagnostics["found_by_cpc"]
].copy()

print(
    "Applications retrieved only by keywords:",
    keyword_only["appln_id"].nunique(),
)

display(
    keyword_only[
        ["appln_id", "appln_title", "retrieval_strategy"]
    ].head(20)
)

,retrieval_strategy,n_applications
0,Keywords,1243
1,CPC,1213
2,Keywords + CPC,884
3,Keywords + IPC + CPC,57
4,IPC + CPC,36
5,IPC,17
6,Keywords + IPC,16


Applications retrieved only by keywords: 1243


,appln_id,appln_title,retrieval_strategy
0,24746,MULTI-ROTOR WIND TURBINE SUPPORTED BY CONTINUO...,Keywords
3,2755148,Multi-rotor wind turbine supported by continuo...,Keywords
4,4305581,MONITORING SYSTEM FOR POWER LINES AND RIGHT-OF...,Keywords
5,5009380,MULTI-ROTOR WIND TURBINE SUPPORTED BY CONTINUO...,Keywords
7,7611249,Wind motor,Keywords
8,13609024,Windenergieanlage Multi-Rotor-Prinzip,Keywords
11,14971374,Method of monitoring land or terrain and deter...,Keywords
12,16419618,Enhanced passive coherent location techniques ...,Keywords
13,19115737,Hydrodynamic or streamlined profile for formin...,Keywords
14,21513231,Multi-rotor wind turbine,Keywords


In [15]:
# INSPECT AN APPLICATION AND ITS RETRIEVAL EVIDENCE
def inspect_application(application_id):
    rows = df_diagnostics.loc[
        df_diagnostics["appln_id"] == application_id
    ]

    if rows.empty:
        print("Application not found in the retrieved dataset.")
        return

    # Retrieve the classification symbols that matched the search queries.
    # These are not necessarily all the codes assigned to the application.
    ipc_codes = sorted(
        df_ipc.loc[
            df_ipc["appln_id"] == application_id,
            "ipc_class_symbol",
        ].dropna().unique().tolist()
    )

    cpc_codes = sorted(
        df_cpc.loc[
            df_cpc["appln_id"] == application_id,
            "cpc_class_symbol",
        ].dropna().unique().tolist()
    )

    # Inspect every text record if the application appears on multiple rows.
    for _, row in rows.iterrows():
        title = (
            str(row["appln_title"])
            if pd.notna(row["appln_title"])
            else ""
        )
        abstract = (
            str(row["appln_abstract"])
            if pd.notna(row["appln_abstract"])
            else ""
        )

        # Reproduce the current case-insensitive substring matching.
        # This also reveals matches occurring inside longer words.
        title_matches = sorted({
            term for term in DRONE_KEYWORDS
            if term.lower() in title.lower()
        })

        abstract_matches = sorted({
            term for term in DRONE_KEYWORDS
            if term.lower() in abstract.lower()
        })

        print("Application ID:", application_id)
        print("DOCDB family ID:", row["docdb_family_id"])
        print("Retrieved by:", row["retrieval_strategy"])
        print("Keywords in title:", title_matches)
        print("Keywords in abstract:", abstract_matches)
        print("Matched IPC codes:", ipc_codes)
        print("Matched CPC codes:", cpc_codes)
        print("\nTITLE:\n", title or "[Not available]")
        print("\nABSTRACT:\n", abstract or "[Not available]")
        print()


# Example: inspect the wind-turbine application shown in the saved results.
#inspect_application(51564414)

In [16]:
# GROUP THE RETRIEVED APPLICATIONS BY DOCDB FAMILY

# Keep applications with a valid family identifier.
valid_family = (
    df_diagnostics["docdb_family_id"].notna()
    & (df_diagnostics["docdb_family_id"] > 0)
)

df_family_hits = df_diagnostics.loc[valid_family].copy()

# Create one row per family.
# These counts refer only to retrieved applications, not all family members.
df_families = (
    df_family_hits
    .groupby("docdb_family_id", as_index=False)
    .agg(
        n_matched_applications=("appln_id", "nunique"),
        matched_application_ids=(
            "appln_id",
            lambda ids: sorted(set(ids)),
        ),
        found_by_keywords=("found_by_keywords", "any"),
        found_by_ipc=("found_by_ipc", "any"),
        found_by_cpc=("found_by_cpc", "any"),
    )
    .sort_values("docdb_family_id")
    .reset_index(drop=True)
)

# Reuse the function defined in the diagnostic cell.
# At family level, different members may match different strategies.
df_families["retrieval_strategy"] = df_families.apply(
    get_retrieval_strategy,
    axis=1,
)

print(
    "Retrieved applications with a valid family ID:",
    df_family_hits["appln_id"].nunique(),
)
print("Distinct DOCDB families:", len(df_families))

display(df_families.head(10))

Retrieved applications with a valid family ID: 3466
Distinct DOCDB families: 2469


,docdb_family_id,n_matched_applications,matched_application_ids,found_by_keywords,found_by_ipc,found_by_cpc,retrieval_strategy
0,5221267,1,[7611249],True,False,False,Keywords
1,7942894,1,[13609024],True,False,False,Keywords
2,11074674,2,"[14034049, 54342525]",False,False,True,CPC
3,11459538,1,[52322637],False,False,True,CPC
4,19895696,1,[42023759],True,False,False,Keywords
5,19914776,2,"[42123841, 53662805]",False,False,True,CPC
6,22019089,1,[46566715],False,False,True,CPC
7,23142817,1,[49394130],True,False,True,Keywords + CPC
8,23143245,1,[49394876],True,False,True,Keywords + CPC
9,24390749,1,[52104346],False,False,True,CPC


In [17]:
# SELECT A REPRODUCIBLE RANDOM SAMPLE OF FAMILIES (100)

validation_sample = (
    df_families
    .sample(
        n=min(100, len(df_families)),
        random_state=42,
    )
    .reset_index(drop=True)
)

# Fields to complete during manual validation.
# Suggested relevance labels: relevant, not relevant, uncertain.
validation_sample["relevance"] = ""
validation_sample["reason"] = ""

print("Families selected for validation:", len(validation_sample))

display(
    validation_sample[
        [
            "docdb_family_id",
            "n_matched_applications",
            "retrieval_strategy",
            "relevance",
            "reason",
        ]
    ].head(10)
)

Families selected for validation: 100


,docdb_family_id,n_matched_applications,retrieval_strategy,relevance,reason
0,77478287,1,Keywords + CPC,,
1,68426213,2,CPC,,
2,72422945,1,Keywords + CPC,,
3,72457225,1,Keywords,,
4,65664234,1,Keywords + CPC,,
5,59497411,1,Keywords + CPC,,
6,59144956,1,Keywords + CPC,,
7,62904216,1,Keywords + CPC,,
8,56849600,1,Keywords + CPC,,
9,50708445,2,Keywords,,


In [19]:
# BUILD A CSV FILE FOR THE MANUAL LABELLING OF THE 100 RANDOM FAMILIES
from pathlib import Path

# Use a new filename when changing the query or switching to PROD.
validation_file = Path("outputs/validation_test_v1.csv")
validation_file.parent.mkdir(parents=True, exist_ok=True)

label_columns = ["docdb_family_id", "relevance", "reason"]

# Reload previous assessments, if available.
if validation_file.exists():
    saved_labels = pd.read_csv(
        validation_file,
        keep_default_na=False,
        dtype={"docdb_family_id": "int64"},
    )

    # Avoid mixing assessments from different samples.
    current_ids = set(validation_sample["docdb_family_id"])
    saved_ids = set(saved_labels["docdb_family_id"])

    if (
        current_ids != saved_ids
        or saved_labels["docdb_family_id"].duplicated().any()
    ):
        raise ValueError(
            "The saved file belongs to a different sample or contains "
            "duplicate family IDs. Use a new validation filename."
        )

    saved_labels = saved_labels.set_index("docdb_family_id")

    for column in ["relevance", "reason"]:
        validation_sample[column] = (
            validation_sample["docdb_family_id"]
            .map(saved_labels[column])
        )

    print("Previous assessments loaded.")

else:
    validation_sample[label_columns].to_csv(
        validation_file,
        index=False,
    )
    print("New validation file created.")


def label_family(family_id, relevance, reason):
    """Record one manual assessment and save all sample labels."""
    allowed_labels = {"relevant", "not relevant", "uncertain"}

    if relevance not in allowed_labels:
        raise ValueError(
            "Use 'relevant', 'not relevant', or 'uncertain'."
        )

    if not reason.strip():
        raise ValueError("Provide a short explanation.")

    mask = validation_sample["docdb_family_id"] == family_id

    if not mask.any():
        raise ValueError("Family not found in the validation sample.")

    validation_sample.loc[mask, "relevance"] = relevance
    validation_sample.loc[mask, "reason"] = reason

    validation_sample[label_columns].to_csv(
        validation_file,
        index=False,
    )

    completed = validation_sample["relevance"].ne("").sum()

    print(f"Assessment saved for family {family_id}.")
    print(f"Completed: {completed}/{len(validation_sample)}")


print("Validation file:", validation_file)

New validation file created.
Validation file: outputs/validation_test_v1.csv


In [18]:
# INSPECT THE RETRIEVED APPLICATIONS OF ONE FAMILY

def inspect_family(family_id):
    applications = (
        df_family_hits.loc[
            df_family_hits["docdb_family_id"] == family_id,
            "appln_id",
        ]
        .drop_duplicates()
        .sort_values()
    )

    if applications.empty:
        print("Family not found in the retrieved dataset.")
        return

    print("DOCDB family:", family_id)
    print("Retrieved applications:", len(applications))
    print()

    # Reuse the application inspection function.
    # It prints full texts and the evidence explaining each match.
    for application_id in applications:
        inspect_application(application_id)


# Inspect the first family in the random sample.
if not validation_sample.empty:
    inspect_family(
        validation_sample.iloc[0]["docdb_family_id"]
    )

DOCDB family: 77478287
Retrieved applications: 1

Application ID: 556553257
DOCDB family ID: 77478287
Retrieved by: Keywords + CPC
Keywords in title: ['unmanned aerial']
Keywords in abstract: ['unmanned aerial']
Matched IPC codes: []
Matched CPC codes: ['B64C  39/024', 'B64U2101/00', 'B64U2101/30']

TITLE:
 Unmanned aerial vehicle bird repelling device

ABSTRACT:
 The invention discloses an unmanned aerial vehicle bird repelling device, relates to the technical field of unmanned aerial vehicles, and aims to solve the problems of long bird repelling time and slow effect due to the fact that manual patrol and vehicle patrol are adopted in airports generally and multiple persons and multiple devices are allocated for patrol. The unmanned aerial vehicle bird repelling device is characterized in that a second bearing shell body is arranged below a first bearing shell body, a positioning table is arranged above the first bearing shell body, the positioning table and the first bearing shell b

In [20]:
# FUNCTION USED TO UPDATE THE CSV TABLE ONE FAMILY AT TIME
# USE THE FUNCTION ABOVE (inspect family) TO LOOK THE FAMILY DETAILS AND THIS FUNCTION TO DECIDE IF IT'S RELEVANT OR NOT
label_family(
    family_id=int(validation_sample.iloc[0]["docdb_family_id"]),
    relevance="relevant",
    reason="The invention describes technology specifically designed for UAVs."
)
#TODO FOR ALL 100 FAMILIES

Assessment saved for family 77478287.
Completed: 1/100


In [24]:
# RETRIEVE ALL AVAILABLE APPLICATIONS OF THE SELECTED FAMILIES

def retrieve_family_members():
    """
    Retrieve all available applications belonging to the families
    identified by the keyword, IPC and CPC searches.
    """

    # Find the valid DOCDB family IDs associated with matched applications.
    # drone_ids is the application-ID union created earlier in the notebook.
    selected_family_ids = (
        db.query(
            TLS201_APPLN.docdb_family_id.label("docdb_family_id")
        )
        .select_from(TLS201_APPLN)
        .join(
            drone_ids,
            TLS201_APPLN.appln_id == drone_ids.c.appln_id,
        )
        .filter(
            TLS201_APPLN.docdb_family_id.isnot(None),
            TLS201_APPLN.docdb_family_id > 0,
        )
        .distinct()
        .subquery()
    )

    # Retrieve applications by family ID.
    # Do not reapply the keyword or classification filters here:
    # a family member does not need to match the original search.
    q_members = (
        db.query(*APPLN_COLUMNS)
        .select_from(TLS201_APPLN)
        .join(
            selected_family_ids,
            TLS201_APPLN.docdb_family_id
            == selected_family_ids.c.docdb_family_id,
        )
        .distinct()
        .order_by(
            TLS201_APPLN.docdb_family_id,
            TLS201_APPLN.appln_id,
        )
    )

    return patstat.df(q_members)


# Execute the query and preserve the original search results.
df_family_members = retrieve_family_members()

print(
    "Applications directly matched by the search:",
    df_drone["appln_id"].nunique(),
)

print(
    "Total available applications in the selected families:",
    df_family_members["appln_id"].nunique(),
)

print(
    "Applications retrieved through family membership and not the search:",
    df_family_members["appln_id"].nunique()
    - df_drone["appln_id"].nunique(),
)

print(
    "Selected families:",
    df_family_members["docdb_family_id"].nunique(),
)

display(df_family_members.head(10))


Applications directly matched by the search: 3466
Total available applications in the selected families: 3562
Applications retrieved through family membership and not the search: 96
Selected families: 2469


,appln_id,docdb_family_id,appln_auth,appln_nr,appln_kind,appln_filing_date,earliest_filing_year,granted
0,7611249,5221267,CN,98111274,A,1998-05-04,1998,N
1,13609024,7942894,DE,20010668,U,2000-06-15,2000,Y
2,14034049,11074674,DE,60119462,T,2001-09-10,2000,Y
3,54342525,11074674,US,96144701,A,2001-09-25,2000,Y
4,52322637,11459538,US,62669703,A,2003-07-25,2002,Y
5,42023759,19895696,NO,924962,A,1992-12-22,1992,N
6,42123841,19914776,WO,2004000108,W,2004-04-16,2003,Y
7,53662805,19914776,US,82449204,A,2004-04-15,2003,Y
8,46566715,22019089,US,5881587,A,1987-06-05,1987,Y
9,49394130,23142817,US,29662499,A,1999-04-22,1999,Y


In [25]:
# CALCULATE THE EARLIEST PRIORITY YEAR FOR EACH DOCDB FAMILY
# Add a priority_year column to df_families by taking the minimum valid
# earliest_filing_year across all available applications in each family.


# Convert years to numeric values.
# Treat missing values and PATSTAT's unavailable-year code (9999) as missing.
years = pd.to_numeric(
    df_family_members["earliest_filing_year"],
    errors="coerce",
)

members = df_family_members.copy()
members["valid_priority_year"] = years.where(
    years.between(1, 9998)
)

# Calculate one priority year per family using all available members.
family_years = (
    members
    .groupby("docdb_family_id", as_index=False)
    .agg(
        priority_year=("valid_priority_year", "min"),
    )
)

family_years["priority_year"] = (
    family_years["priority_year"].astype("Int64")
)

# Add the priority year to the existing family table.
# Removing a previous copy makes this cell safe to rerun.
df_families = (
    df_families
    .drop(columns=["priority_year"], errors="ignore")
    .merge(
        family_years,
        on="docdb_family_id",
        how="left",
        validate="one_to_one",
    )
)

print("Families:", len(df_families))
print(
    "Families without an available priority year:",
    df_families["priority_year"].isna().sum(),
)

display(
    df_families[
        ["docdb_family_id", "priority_year", "n_matched_applications"]
    ].head(10)
)

Families: 2469
Families without an available priority year: 0


,docdb_family_id,priority_year,n_matched_applications
0,5221267,1998,1
1,7942894,2000,1
2,11074674,2000,2
3,11459538,2002,1
4,19895696,1992,1
5,19914776,2003,2
6,22019089,1987,1
7,23142817,1999,1
8,23143245,1999,1
9,24390749,1932,1


In [26]:
# EXTRACT APPLICANTS OF THE SELECTED FAMILIES
# Include all available family members, not only applications
# that directly matched the keyword or classification searches.

selected_family_ids = (
    df_families["docdb_family_id"]
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)

q_applicants = (
    db.query(
        TLS201_APPLN.docdb_family_id,
        TLS201_APPLN.appln_id,
        TLS206_PERSON.person_id,
        TLS206_PERSON.person_name,
        TLS206_PERSON.psn_name,
        TLS206_PERSON.person_ctry_code,
    )
    .select_from(TLS201_APPLN)
    .join(
        TLS207_PERS_APPLN,
        TLS201_APPLN.appln_id == TLS207_PERS_APPLN.appln_id,
    )
    .join(
        TLS206_PERSON,
        TLS207_PERS_APPLN.person_id == TLS206_PERSON.person_id,
    )
    .filter(
        TLS201_APPLN.docdb_family_id.in_(selected_family_ids),

        # Keep applicants, including persons who are also inventors.
        TLS207_PERS_APPLN.applt_seq_nr > 0,
    )
    .distinct()
    .order_by(
        TLS201_APPLN.docdb_family_id,
        TLS201_APPLN.appln_id,
        TLS206_PERSON.person_id,
    )
)

df_applicants = patstat.df(q_applicants)

print("Application-applicant records:", len(df_applicants))
print(
    "Distinct applicant person IDs:",
    df_applicants["person_id"].nunique(),
)
print(
    "Families with at least one applicant:",
    df_applicants["docdb_family_id"].nunique(),
)

# Report selected families for which no applicant record was retrieved.
families_without_applicants = (
    set(selected_family_ids)
    - set(df_applicants["docdb_family_id"])
)

print(
    "Families without a retrieved applicant:",
    len(families_without_applicants),
)

display(df_applicants.head(10))

Application-applicant records: 4192
Distinct applicant person IDs: 2617
Families with at least one applicant: 2445
Families without a retrieved applicant: 24


,docdb_family_id,appln_id,person_id,person_name,psn_name,person_ctry_code
0,5221267,7611249,18014679,ZHU JIE,ZHU JIE,CN
1,7942894,13609024,13020973,"RUEHLE, WILFRIED","RUEHLE, WILFRIED",DE
2,7942894,13609024,24147543,"DOHM, RUDOLF","DOHM, RUDOLF",DE
3,11074674,14034049,10416418,Rafael-Armament Development Authority Ltd.,RAFAEL-ARMAMENT DEVELOPMENT AUTHORITY,IL
4,11074674,54342525,6083155,"Rafael Armament Development Authority, Ltd.",RAFAEL-ARMAMENT DEVELOPMENT AUTHORITY,IL
5,11459538,52322637,5394522,C.R.F. Societa Consortile per Azioni,CRF (CENTRO RICERCHE FIAT),IT
6,19895696,42023759,33776463,"NORSKE STATS OLJESELSKAP AS, DEN","NORSKE STATS OLJESELSKAP AS, DEN",NO
7,19914776,42123841,12621140,"MUREN, PETTER","MUREN, PETTER",NO
8,19914776,42123841,13156884,PROXFLYER AS,PROXFLYER,NO
9,19914776,53662805,10112626,Proxflyer AS,PROXFLYER,NO


In [27]:
# CLEAN APPLICANT NAMES
# Prefer the PATSTAT standardized name (psn_name) when available.
# Otherwise, use the original person_name.
# Preserve the original DataFrame and its application-person relationships.

df_applicants_clean = df_applicants.copy()


def normalize_names(series):
    """Normalize formatting while preserving meaningful name components."""
    return (
        series
        .astype("string")
        .str.normalize("NFKC")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .str.upper()
        .replace("", pd.NA)
    )


# Normalize both name fields.
df_applicants_clean["person_name_clean"] = normalize_names(
    df_applicants_clean["person_name"]
)

df_applicants_clean["psn_name_clean"] = normalize_names(
    df_applicants_clean["psn_name"]
)

# Use the standardized name, falling back to the original name.
df_applicants_clean["applicant_name"] = (
    df_applicants_clean["psn_name_clean"]
    .fillna(df_applicants_clean["person_name_clean"])
)

# Record the source of the chosen name for traceability.
df_applicants_clean["name_source"] = "missing"

df_applicants_clean.loc[
    df_applicants_clean["person_name_clean"].notna(),
    "name_source",
] = "person_name"

df_applicants_clean.loc[
    df_applicants_clean["psn_name_clean"].notna(),
    "name_source",
] = "psn_name"

# Report data quality without dropping unnamed applicant records.
print("Application-applicant records:", len(df_applicants_clean))

print(
    "Records without an available name:",
    df_applicants_clean["applicant_name"].isna().sum(),
)

print(
    "Distinct normalized names:",
    df_applicants_clean["applicant_name"].nunique(),
)

print("\nName sources:")
display(df_applicants_clean["name_source"].value_counts())

display(
    df_applicants_clean[
        [
            "person_id",
            "person_name",
            "psn_name",
            "applicant_name",
            "name_source",
        ]
    ].head(20)
)

Application-applicant records: 4192
Records without an available name: 0
Distinct normalized names: 2096

Name sources:


name_source
psn_name    4192
Name: count, dtype: int64

,person_id,person_name,psn_name,applicant_name,name_source
0,18014679,ZHU JIE,ZHU JIE,ZHU JIE,psn_name
1,13020973,"RUEHLE, WILFRIED","RUEHLE, WILFRIED","RUEHLE, WILFRIED",psn_name
2,24147543,"DOHM, RUDOLF","DOHM, RUDOLF","DOHM, RUDOLF",psn_name
3,10416418,Rafael-Armament Development Authority Ltd.,RAFAEL-ARMAMENT DEVELOPMENT AUTHORITY,RAFAEL-ARMAMENT DEVELOPMENT AUTHORITY,psn_name
4,6083155,"Rafael Armament Development Authority, Ltd.",RAFAEL-ARMAMENT DEVELOPMENT AUTHORITY,RAFAEL-ARMAMENT DEVELOPMENT AUTHORITY,psn_name
5,5394522,C.R.F. Societa Consortile per Azioni,CRF (CENTRO RICERCHE FIAT),CRF (CENTRO RICERCHE FIAT),psn_name
6,33776463,"NORSKE STATS OLJESELSKAP AS, DEN","NORSKE STATS OLJESELSKAP AS, DEN","NORSKE STATS OLJESELSKAP AS, DEN",psn_name
7,12621140,"MUREN, PETTER","MUREN, PETTER","MUREN, PETTER",psn_name
8,13156884,PROXFLYER AS,PROXFLYER,PROXFLYER,psn_name
9,10112626,Proxflyer AS,PROXFLYER,PROXFLYER,psn_name
